# KernelSynth: cobertura por AR e pelo vocabulário de mecanismos

Esta avaliação substitui a pergunta “ARIMA prevê bem?” por **“quanto dos padrões do KernelSynth conseguimos representar e gerar com nosso vocabulário, sob limites explícitos de operações e lags?”**. Os notebooks anteriores são resultados históricos preservados. TimePFN/LMC permanece nesses estudos; este novo protocolo trata KernelSynth univariado.

Uma boa previsão de um passo não demonstra equivalência de geradores. Por isso, medimos três objetos: previsão com histórico observado, continuação recursiva sem observar o futuro e geração estocástica com inovações independentes. Não prometemos cobertura universal: a suíte tem os 33 itens do banco e 20 composições fixadas por seed, não todas as composições nem todos os dados usados no treinamento do Chronos.

## O que o KernelSynth gera

O artigo de Chronos, §4.2 e Algoritmo 2 ([ansari2024chronos](https://arxiv.org/html/2403.07815v1#S4.SS2), chave em `references/library.bib`), define amostras de um GP de média zero, cujo kernel é composto por soma e produto. A implementação oficial fixa o banco e o comprimento; usamos a [revisão imutável do script](https://github.com/amazon-science/chronos-forecasting/blob/069df04e012f6014729246a718067dd3e541480e/scripts/kernel-synth.py), banco nas linhas 21–56 e geração nas linhas 142–171. Não confundimos esse banco com funções determinísticas.

**Todas essas famílias são aleatórias**, mesmo quando suas trajetórias são suaves. Nenhuma é retirada para melhorar a taxa de sucesso. O ruído branco fica no diagnóstico: sua média condicional é zero, mas seu erro de previsão realizado é inevitável. O GP privilegiado abaixo quantifica essa diferença também nos kernels compostos.

## Hipóteses e limites que realmente testamos

1. Um AR esparso já pode representar boa parte da dependência previsível.
2. Uma busca no vocabulário pode encontrar expressões compactas; aumentar o orçamento pode ajudar, mas a heurística e a estimação podem falhar.
3. Bom ajuste com histórico observado pode não sobreviver à geração recursiva.

O dicionário busca lags, todos os produtos quadráticos de dois lags (incluindo quadrados) e `tanh(lag)`, combinados por soma e coeficientes. É **um subconjunto da gramática**, sem produtos de ordem superior, `tanh` aninhado ou busca exaustiva de topologias. O AR usa apenas os termos lineares, pelo mesmo algoritmo. A seleção gulosa maximiza a redução do erro de treino e escolhe o melhor prefixo pela validação. Não é garantido que a busca não linear reencontre o melhor AR.

Cada termo linear custa duas operações (multiplicação pelo parâmetro e soma); cada termo quadrático ou `tanh` custa três. Contamos a árvore exportada, incluindo todos os coeficientes. O intercepto é um terminal. Com orçamento 6, por exemplo, cabem no máximo três lags lineares com coeficientes. Os limites do gerador básico são 6 operações e lag 12; testamos também 14 e 24, cruzados independentemente.

Os coeficientes ajustados são livres. Isso mede expressividade do vocabulário, **não** cobertura pelo amostrador padrão de parâmetros de magnitude 0,05–0,90, nem certificação de contração/estacionariedade. `tanh` pertence à representação e aos mecanismos não lineares, mas não ao amostrador binário básico. Dados são divididos pelo desvio-padrão agregado do treino; árvores e parâmetros registrados operam nessas unidades.

## Métricas e separação da aleatoriedade

Usamos $\mathrm{NRMSE}=\sqrt{\operatorname{mean}((y-\hat y)^2)}/s_{\mathrm{treino}}$. O denominador é a escala de amplitude, não o comprimento: a média já normaliza a quantidade de pontos. Isso evita que séries mais longas pareçam artificialmente mais fáceis. O desvio-padrão é agregado sobre as realizações de treino, inclusive para o kernel constante, que varia entre realizações.

Reportamos limiares **0,001, 0,01 e 0,1**, sem escolhê-los pelo teste. 0,1 é uma referência exploratória de RMSE de 10% da escala (MSE de 1% da variância); 0,001 exige MSE de um milionésimo da variância. Não existe um limiar universal que prove equivalência.

Para cada tempo, o oráculo GP conhece a covariância e calcula $m_t=K_{t,H}K_{H,H}^{+}x_H$ e $v_t=K_{tt}-K_{t,H}K_{H,H}^{+}K_{H,t}$, com **o mesmo limite de lag**. Medimos tanto o erro contra a observação quanto contra $m_t$ e o piso esperado $\sqrt{\operatorname{mean}(v_t)}/s_{\mathrm{treino}}$. Condicionalmente a um kernel fixo, essa média é linear no histórico; seus coeficientes podem variar no tempo. O oráculo não é um AR homogêneo treinável nem recebe kernels misturados sem identificação. A identidade gaussiana implica $E[(Y-f(H))^2]=E[v(H)]+E[(m(H)-f(H))^2]$; portanto não exigimos que uma inovação nova reproduza o sorteio original.

A continuação de 64 passos usa somente o prefixo observado. Na geração longa, fornecemos 24 valores iniciais e depois apenas saídas geradas e ruído gaussiano IID, com escala estimada dos resíduos de treino. Essa lei de inovação simples é outra limitação: resíduos podem ser heteroscedásticos ou correlacionados. Divergências viram falhas explícitas; não recortamos valores nem descartamos trajetórias.

ACF, espectro, variância e Wasserstein comparam ensembles gerados e de teste. Outro ensemble independente do **mesmo GP**, de tamanho igual, mostra o nível de discrepância amostral dessas métricas. Não são testes formais de igualdade, não provam igualdade das distribuições conjuntas e a calibração única não é um intervalo de confiança. As realizações de teste são independentes do treino e da validação; os tempos avaliados são comuns a todos os modelos. A avaliação mede transferência a novas realizações do mesmo kernel no mesmo domínio temporal, não extrapolação para outro domínio.

In [ ]:
from pathlib import Path
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/tsfmxai').exists())
sys.path.insert(0, str(ROOT / 'src'))
import hashlib, json, platform, subprocess, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from threadpoolctl import threadpool_limits
from tsfmxai.pipelines.kernel_vocabulary_coverage import CoverageConfig, run_coverage_study
from tsfmxai.evaluation.process_coverage import DEFAULT_THRESHOLDS
CONFIG = CoverageConfig()
THRESHOLDS = [0.001, 0.01, 0.1]
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_colwidth', 90)
print(CONFIG)


In [ ]:
with threadpool_limits(limits=1):
    study = run_coverage_study(CONFIG, progress=lambda i, n, name: print(f'{i}/{n}: {name}') if i % 10 == 0 or i == n else None)
results = pd.DataFrame(study['rows'])
assert len(results) == (33 + CONFIG.composition_count) * len(CONFIG.lags) * len(CONFIG.operators) * 2
assert (results.used_operators <= results.max_operators).all()
assert (results.used_max_lag <= results.max_lag).all()
print(f'{len(results)} avaliações; nenhum kernel excluído.')


## Resultados por orçamento

Cada célula da tabela resume 53 casos de peso igual (incluindo itens periódicos duplicados do banco oficial). Medianas não significam cobertura de todos os casos. Os quantis entre trajetórias são descritivos, não intervalos de confiança. Não selecionamos o orçamento vencedor no teste.

In [ ]:
group = ['model', 'max_lag', 'max_operators']
summary = results.groupby(group).agg(
    cases=('case', 'count'), one_step_median=('one_step_nrmse', 'median'),
    oracle_mean_median=('oracle_mean_nrmse', 'median'),
    rollout_median=('rollout_nrmse', 'median'),
    divergent_fraction_mean=('divergent_fraction', 'mean'))
coverage = []
for keys, frame in results.groupby(group):
    for threshold in THRESHOLDS:
        coverage.append(dict(zip(group, keys), threshold=threshold,
            one_step_pass=int((frame.one_step_nrmse <= threshold).sum()),
            oracle_mean_pass=int((frame.oracle_mean_nrmse <= threshold).sum()),
            rollout_pass=int((frame.rollout_nrmse <= threshold).sum()), total=len(frame)))
coverage = pd.DataFrame(coverage)
display(summary)
display(coverage)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 14), constrained_layout=True)
for ax, metric, title in zip(axes, ['one_step_nrmse', 'oracle_mean_nrmse'],
                            ['Erro contra a observação', 'Erro contra a média condicional GP']):
    table = results.pivot(index='case', columns=group, values=metric)
    image = ax.imshow(np.log10(np.maximum(table.to_numpy(), 1e-6)), aspect='auto', vmin=-4, vmax=1, cmap='viridis')
    ax.set_yticks(range(len(table)), table.index, fontsize=7)
    ax.set_xticks(range(len(table.columns)), [f'{m}\nlag {l}, ops {o}' for m,l,o in table.columns], rotation=90, fontsize=8)
    ax.set_title(title)
fig.colorbar(image, ax=axes, label='log10 NRMSE; −3 = 0,001; −1 = 0,1', shrink=0.6)
plt.show()


## Geração e variabilidade

A tabela abaixo mostra todos os casos no orçamento ampliado. `oracle_mean_nrmse` pequeno com `one_step_nrmse` alto é compatível com inovação imprevisível. A coluna recursiva pode divergir mesmo quando a previsão de um passo é boa. A tabela de distâncias inclui a comparação GP–GP; suas escalas não são intercambiáveis.

In [ ]:
expanded = results[(results.max_lag == max(CONFIG.lags)) & (results.max_operators == max(CONFIG.operators))]
display(expanded[['case', 'model', 'used_operators', 'used_max_lag', 'one_step_nrmse',
                  'oracle_mean_nrmse', 'irreducible_nrmse', 'rollout_nrmse', 'divergent_fraction']].reset_index(drop=True))
distance_columns = list(DEFAULT_THRESHOLDS)
display(expanded.groupby('model')[distance_columns + ['gp_calibration_'+k for k in distance_columns]].median())
path_spread = expanded[['case', 'model']].copy()
for q in [0.1, 0.5, 0.9]:
    path_spread[f'path_q{q}'] = expanded.per_path_one_step_nrmse.map(lambda x: np.quantile(x, q))
display(path_spread.reset_index(drop=True))


In [ ]:
chosen_cases = ['periodic_01_p24', 'rbf_scale0.1', 'white_level1', 'composite_01']
fig, axes = plt.subplots(len(chosen_cases), 2, figsize=(14, 11), constrained_layout=True)
for row, case in enumerate(chosen_cases):
    reference = study['examples'][(case, 'sparse_AR')]['reference']
    for ax in axes[row]:
        ax.plot(reference, color='black', linewidth=1, alpha=0.65, label='KernelSynth: teste')
    for model in ['sparse_AR', 'AST_dictionary']:
        ex = study['examples'][(case, model)]
        axes[row, 0].plot(np.arange(CONFIG.length-CONFIG.horizon, CONFIG.length), ex['forecast'], label=model)
        axes[row, 1].plot(ex['generated'], label=model, alpha=0.75)
    axes[row, 0].set_xlim(CONFIG.length-2*CONFIG.horizon, CONFIG.length)
    axes[row, 0].set_title(case + ': continuação de 64 passos')
    axes[row, 1].set_title(case + ': geração com inovação independente')
    for ax in axes[row]:
        ax.set_ylim(-6, 6)
        ax.set_ylabel('Unidades do desvio-padrão de treino')
        ax.legend(fontsize=7)
plt.show()
print('Limites visuais ±6 apenas nos gráficos; erros e divergências usam os valores completos.')
display(expanded[['case', 'model', 'formula']].reset_index(drop=True))


## Um mecanismo além da lei do KernelSynth

Não é necessário procurar uma curva visualmente impossível. Um exemplo analítico no nosso vocabulário é

$$x_t=0.5\tanh(x_{t-1})+\varepsilon_t,\qquad \varepsilon_t\sim\mathrm{Uniform}(-0.5,0.5),\quad x_0=0.$$

O mecanismo usa `Multiply`, `Parameter`, `Tanh` e `Lag` (duas operações), com a mesma família de inovação uniforme implementada em `generators/contraction.py`. Como $|\tanh(x)|<1$, todas as saídas estão em $(-1,1)$, e a inovação tem variância positiva. Já uma marginal gaussiana de variância positiva tem suporte não limitado. Uma mistura dos GPs de média zero do KernelSynth também atribui probabilidade positiva fora de qualquer intervalo limitado sempre que há massa em componentes de variância positiva; se todas as variâncias fossem zero, produziria apenas zero. Logo **essa lei limitada e não degenerada não é exatamente a lei do KernelSynth**. A derivada da parte determinística é limitada por 0,5, oferecendo ainda contração.

Essa é uma derivação sobre distribuições, não uma afirmação de que o KernelSynth jamais possa produzir uma trajetória numericamente próxima em uma janela finita. Também não prova a outra direção, isto é, que nosso gerador com orçamento finito contém todos os GPs. As duas afirmações devem permanecer separadas na tese.

## Registro completo do experimento

A célula seguinte é o registro único: pergunta, hipótese, configuração, fontes, ambiente, hashes, seeds, manifesto, métricas por caso, fórmulas com parâmetros completos, síntese científica e comando de reprodução. Não há arquivos auxiliares de resultados. Os números são evidência desta suíte finita, com uma seed de composição e os tamanhos de ensemble registrados.

In [ ]:
# REGISTRO COMPLETO DO EXPERIMENTO — fonte única de configuração, métricas e proveniência
source_paths = [
    'src/tsfmxai/symbolic_regression/kernel_coverage.py',
    'src/tsfmxai/pipelines/kernel_vocabulary_coverage.py',
    'src/tsfmxai/generators/kernel_synth.py',
    'src/tsfmxai/evaluation/process_coverage.py',
    'src/tsfmxai/representations/expression.py',
]
strict = coverage[(coverage.threshold == 0.001) & (coverage.max_lag == max(CONFIG.lags)) & (coverage.max_operators == max(CONFIG.operators))]
practical = coverage[(coverage.threshold == 0.1) & (coverage.max_lag == max(CONFIG.lags)) & (coverage.max_operators == max(CONFIG.operators))]
sentences = []
for _, row in practical.iterrows():
    sentences.append(f"{row['model']}: NRMSE ≤ 0,1 em {row.one_step_pass}/{row.total} casos de um passo, {row.oracle_mean_pass}/{row.total} contra a média GP e {row.rollout_pass}/{row.total} em continuação recursiva.")
for _, row in strict.iterrows():
    sentences.append(f"{row['model']}: no limiar 0,001, {row.one_step_pass}/{row.total} casos de um passo e {row.oracle_mean_pass}/{row.total} contra a média GP.")
sentences.append('Não foi demonstrada inclusão universal do KernelSynth. Falha da busca finita não prova impossibilidade de representação; êxito preditivo não prova equivalência da geração.')
record = {
    'question': 'Qual cobertura empírica do KernelSynth é atingida por AR e busca no vocabulário sob limites de operações e lags?',
    'hypotheses': ['AR pode representar grande parte da dependência previsível', 'Maior orçamento pode ajudar', 'Previsão de um passo não garante geração fiel'],
    'configuration': study['config'],
    'fixed_protocol': {'thresholds': THRESHOLDS, 'scale': 'pooled training standard deviation, no centering',
        'common_scoring_start': max(CONFIG.lags), 'search': 'greedy forward least squares, validation-selected prefix',
        'dictionary': ['lag', 'lag*lag including squares', 'tanh(lag)'],
        'coefficient_bounds': 'unrestricted; not default random-sampler support',
        'operation_costs': {'linear_term': 2, 'quadratic_or_tanh_term': 3, 'intercept': 0},
        'numeric': {'least_squares_rcond': 1e-10, 'oracle_pinv_rcond': 1e-12, 'feature_residual_norm_cutoff': 1e-16,
                    'divergence_absolute_limit': 1e6, 'blas_threads': 1, 'acf_lags': 16},
        'sampling': 'stable_factor; oracle uses actual factor @ factor.T including numerical stabilization',
        'sampling_jitter': 'Cholesky tries 10**exponent * max(mean(diagonal(K)),1), exponent=-10,...,-4; eigen fallback floors at 1e-8 * scale',
        'composition_sampling': '20 retained compositions, 2 + case_number % 4 kernels; uniform bank indices with replacement and uniform +/*; not official random length frequencies',
        'split': 'independent realizations: train, validation, test, GP calibration, in that order',
        'innovation': 'Gaussian IID; training residual RMS; independent seed; same stream across models for paired comparison',
        'distribution_metrics': 'ACF MAE, PSD Jensen-Shannon (natural log), absolute log variance ratio, standardized Wasserstein; descriptive GP-GP calibration'},
    'seeds': {'composition': CONFIG.seed+10000, 'samples': 'seed+100000+case_index',
              'generation': 'seed+200000+case_index', 'optimizer': 'deterministic, no RNG'},
    'provenance': {'date': '2026-10-30', 'bibliography_key': 'ansari2024chronos',
        'paper': 'https://arxiv.org/html/2403.07815v1#S4.SS2', 'evidence': 'section 4.2 and Algorithm 2; official script lines 21–56, 142–171',
        'official_source': 'https://github.com/amazon-science/chronos-forecasting/blob/069df04e012f6014729246a718067dd3e541480e/scripts/kernel-synth.py',
        'git_head': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip(),
        'source_sha256': {p: hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in source_paths},
        'working_tree_note': 'Existing unrelated working-tree changes preserved; source hashes identify the actual implementation used.',
        'python': sys.version, 'platform': platform.platform(), 'processor': platform.processor(), 'cpu_count': os.cpu_count(),
        'versions': {p: importlib.metadata.version(p) for p in ['numpy','scipy','scikit-learn','pandas','matplotlib','nbformat','nbclient','threadpoolctl']}},
    'manifest': study['manifest'], 'metrics_and_exact_bindings': study['rows'],
    'coverage_counts': coverage.to_dict(orient='records'), 'scientific_summary': sentences,
    'limitations': ['Finite suite and finite heuristic dictionary', 'Small ensembles; no formal uncertainty interval',
        'No stationarity certificate or coefficient-prior constraint', 'No exhaustive composite coverage',
        'No claim of equality of stochastic process laws', 'TimePFN/LMC not rerun in this protocol'],
    'reproduce_from_repository_root': r'.\.venv\Scripts\python.exe -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 experiments/autoregressive_process_coverage/vocabulary_coverage/01_kernelsynth_vocabulary_coverage.ipynb',
}
# JSON standard: divergent scores are explicitly encoded as strings, never silently omitted.
def json_safe(value):
    if isinstance(value, dict): return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)): return [json_safe(v) for v in value]
    if isinstance(value, float) and not np.isfinite(value): return str(value)
    return value
display(Markdown('**Síntese científica desta execução**\n\n' + '\n\n'.join(sentences)))
print(json.dumps(json_safe(record), ensure_ascii=False, indent=2, allow_nan=False))
